<a href="https://colab.research.google.com/github/soumya24sahu/-ML-Based-Equalizer-for-BER-Improvement-in-Long-Haul-Optical-Fiber-Communication/blob/main/ML_based_equalizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Import the 'drive' module from Google Colab, which allows access to Google Drive
from google.colab import drive
# Mount Google Drive to the Colab environment at the specified path
# This lets you read/write files in your Google Drive as if they were local files
drive.mount('/content/drive')

In [ ]:
import numpy as np

# Path to your saved files
# 'data_path' points to the .npy file containing the input data (e.g., signals/features)
data_path = "/content/drive/MyDrive/Project/test/5_240/data.npy"
# 'label_path' points to the .npy file containing the corresponding labels/targets
label_path = "/content/drive/MyDrive/Project/test/5_240/label.npy"

# Load data
# np.load() reads the .npy binary file and reconstructs it as a NumPy array
data = np.load(data_path)   # Load the input data array , to train a model
label = np.load(label_path) # Load the corresponding label array ,The model compares the output it generates with the label to check whether it is correct or not.

# Print the shapes to verify the data loaded correctly and to confirm dimensions match what you expect (e.g., samples x features)
print("Data shape:", data.shape)
print("Label shape:", label.shape)

In [ ]:
import tensorflow as tf
from optic.dsp.core import pulseShape, firFilter, decimate, symbolSync, pnorm, signalPower
from optic.utils import parameters
import numpy as np
from optic.plot import pconst, plotPSD



In [ ]:
Rs  = 5e9         # symbol rate [baud]
SpS = 2           # samples per symbol

In [ ]:
# FIR FILTER DESIGN TO COMPENSATE FOR CHROMATIC DISPERSION


from numpy import pi, exp, sqrt, sin      # bring specific math functions into local namespace
from scipy import linalg, special          # linalg.solve -> solve linear systems; special.erf -> complex error function
from scipy.fft import ifft                 # inverse FFT (imported but unused in this snippet)
import warnings                            # used to raise non-fatal symmetry warnings
import numpy as np


class cd_fir_filter:
    """Finite impulse response (FIR) filter for compensating chromatic dispersion (CD) using the least squares method.

    Chromatic dispersion is modeled as:
  EDC(w) = exp(-j beta/2 * L * w^2)
where EDC(w) is the ideal frequency-domain response for compensating dispersion.
 Args (provided as a dictionary):
        beta2: dispersion parameter
        fsamp: sampling frequency in Hz
        bandwidth: percentage of compensated bandwidth, between 0 and 1, defaults to 1
        eps: regularization parameter, defaults to 1e-12
    """

    def __init__(self, fsamp, Rs, rolloff=0.01):

        # ADC/DSP sampling frequency in Hz (e.g. 2 samples/symbol for coherent Rx)
        self.fsamp = fsamp

        # Fixed group-velocity dispersion parameter for standard SMF near 1550 nm
        # units: s^2/m  (≈ -21 ps^2/km, the typical textbook value for SSMF)
        self.beta2 = -20.47 * 1.0e-27

        # RRC pulse-shaping roll-off factor (excess bandwidth), default 1%
        self.rolloff = rolloff

        # Normalized signal bandwidth as a fraction of fsamp:
        # (1+rolloff)*Rs = occupied bandwidth of the RRC-shaped signal (Hz)
        # dividing by fsamp expresses it on a [0, 1] scale for filter design
        self.bandwidth = (1 + rolloff) * Rs / self.fsamp

        # Small regularization constant added to the diagonal of Q later,
        # to keep the least-squares matrix well-conditioned (avoid near-singularity)
        self.eps = 1e-12

        # Sanity check: normalized bandwidth must lie in [0, 1]
        if not (0.0 <= self.bandwidth <= 1.0):
            raise ValueError(f"bandwidth parameter must be between 0 and 1: bandwidth = {self.bandwidth}")

    def get_required_filter_length(self, L):
        # Allow L to be scalar or array (fiber length in meters)
        L = np.array(L)

        # Absolute occupied bandwidth in Hz (de-normalize self.bandwidth)
        Df = self.bandwidth * self.fsamp

        # Ip & Kahn-style formula: minimum FIR length needed so the impulse
        # response captures the full CD-induced pulse spread over length L,
        # forced to be odd (2*floor(...) + 1) so there's a single center tap
        return (2 * np.floor(2 * pi * np.abs(self.beta2) * L * Df * self.fsamp / 2) + 1).astype(np.int32)

    def get_filter(self, L, cd_filter_length):
        """ Approximates exp(j*KK*w^2), where w=-pi..pi, using least squares.
        Args:
            L: fiber length [m]
            cd_filter_length: length of the FIR filter, must be odd
        Returns:
            Filter coefficients as a numpy array
        """
        # Enforce odd length -> guarantees a well-defined center tap for symmetry
        if cd_filter_length % 2 == 0:
            raise ValueError(f"cd_filter_length must be odd: cd_filter_length={cd_filter_length}")

        # Combined dispersion-length constant, expressed per-sample instead of per-Hz
        # (scaled by fsamp^2); negative sign of beta2 flips it since we want the
        # *compensating* (inverse-dispersion) response
        KK = -(self.beta2) / 2 * L * (self.fsamp**2)

        # Number of taps on each side of the center tap
        # (total filter length = 2*delay + 1)
        delay = (cd_filter_length - 1) // 2

        # Normalized bandwidth over which the ideal response is matched
        xi = self.bandwidth

        # Q: least-squares "design" matrix — a sinc-based autocorrelation
        # of the ideal band-limited response, size (cd_filter_length x cd_filter_length)
        Q = np.ones([cd_filter_length, cd_filter_length])  # diagonal stays 1 (sinc(0) = 1)
        for m in range(cd_filter_length):
            for n in range(cd_filter_length):
                if m != n:  # off-diagonal entries follow sinc(xi*(m-n))
                    Q[m, n] = sin(pi * (m - n) * xi) / (pi * (m - n) * xi)
        Q = xi * Q   # scale whole matrix by bandwidth

        # Tap indices centered at 0: -delay, ..., 0, ..., +delay
        nn = np.arange(-delay, delay + 1)

        # Desired/target vector: analytically integrated ideal CD response
        # evaluated at each tap position, scaled by bandwidth
        v = xi * self.int_aux(nn, KK, xi)

        # Solve regularized normal equations (Q + eps*I) * h = v for filter taps h
        # Regularization prevents instability if Q is nearly singular
        cd_filter_coeffs = linalg.solve(Q + self.eps * np.eye(cd_filter_length), v)

        # Sanity check: coefficients should be symmetric (CD compensation is an
        # even, linear-phase-like operation), so compare mirrored tap pairs
        for i in range(delay):
            absdiff = abs(cd_filter_coeffs[i] - cd_filter_coeffs[cd_filter_length - i - 1])
            if absdiff > 1e-2:
                warnings.warn(f"Filter coefficients are not symmetric: absolute difference = {absdiff}")

        # Since symmetric, only need center-tap-to-last-tap half of coefficients
        tmp = cd_filter_coeffs.reshape([cd_filter_length])[delay:]

        # Reshape to (num_taps, 1, 1) — likely matches shape expected for
        # broadcasting/convolution downstream (e.g. per-polarization application)
        tmp = np.reshape(tmp, (-1, 1, 1))

        # Return real and imaginary parts separately (coefficients are complex;
        # splitting is common for real/imag-separated FIR hardware implementations)
        return np.real(tmp), np.imag(tmp)

    def int_aux(self, n, KK, x):
        """ Auxiliary function for integration approximation.

        Args:
            n: index array
            KK: dispersion constant
            x: bandwidth scaling factor

        Returns:
            Integrated values as numpy array
        """
        # Lower and upper angular-frequency bounds of the normalized band [-pi*x, pi*x]
        Omega1 = -pi * x
        Omega2 = pi * x

        # Closed-form solution to the integral of exp(j*KK*w^2)*exp(-j*n*w) dw
        # over w in [Omega1, Omega2]. Obtained by completing the square and
        # expressing the resulting Fresnel-type integral via the complex erf function
        y = exp(-1j * (n**2 / (4 * KK) + 3 * pi / 4)) / (2 * (Omega2 - Omega1)) * sqrt(pi / KK + 0j) \
            * (special.erf(exp(-1j * pi / 4) * (2 * Omega1 * KK + n) / (2 * sqrt(KK + 0j))) \
               - special.erf(exp(-1j * pi / 4) * (2 * Omega2 * KK + n) / (2 * sqrt(KK + 0j))))
        return y

In [ ]:
sampfreq = Rs * SpS   # Sampling frequency = symbol rate × oversampling factor (samples per symbol)
                       # e.g. Rs=32e9 Baud, SpS=2  ->  sampfreq = 64e9 Sa/s (64 GSa/s)

cdfilter = cd_fir_filter(sampfreq, Rs)   # Instantiate the CD-compensation FIR filter designer
                                        # rolloff uses the class default (0.01) since it isn't passed

print(sampfreq)   # Sanity check: confirm the computed sampling rate is what you expect

In [ ]:
def calculate_step_sizes(Lsp, Nsp, StPS, alpha_lin=0.046, direction=-1, adjusting_factor=0.4):
    """
    Calculate the dispersion (cd_length) and nonlinearity (nl_length) step sizes
    for the symmetric SSFM with combined half-steps, based on per-span length.
    """

    # Scale the fiber's linear attenuation coefficient by a tunable factor.
    # This lets you "soften" or "sharpen" how aggressively step sizes shrink/grow
    # without changing the physical alpha used elsewhere in the simulation.
    alpha_adj = adjusting_factor * alpha_lin

    # Total normalized "power loss budget" over one span:
    #   (1 - exp(-alpha_adj * Lsp))  is the fraction of power lost across the whole span
    #   (via integrating the exponential decay curve).
    # Dividing by StPS splits that loss budget into StPS EQUAL-sized chunks.
    # Each chunk will correspond to one nonlinear step.
    delta = (1 - np.exp(-alpha_adj * Lsp)) / StPS
    step_size = np.zeros(StPS)  # placeholder (overwritten below)

    # Decide the order in which step indices are generated.
    # direction = -1 ("backward"): nn = 1, 2, ..., StPS
    # direction =  1 ("forward"):  nn = StPS, ..., 2, 1
    # This flips whether the resulting step_size array runs
    # small-step-first or large-step-first.
    if direction == -1:
        nn = np.arange(1, StPS + 1)          # Backward direction: 1, 2, ..., StPS
    else:
        nn = np.arange(StPS, 0, -1)          # Forward direction: StPS, ..., 2, 1

    # Core formula: invert the exponential decay curve to find the physical
    # length of the step whose *normalized power loss* equals one delta-chunk.
    # (StPS - nn) plays the role of the "chunk index" k = 0 ... StPS-1.
    # This is the standard closed-form solution for step length z such that
    # the integral of exp(-alpha*z) over the step equals `delta`.
    step_size = -1 / (alpha_adj) * np.log(
        (1 - (StPS - nn + 1) * delta) / (1 - (StPS - nn) * delta)
    )
    # Result: step_size holds StPS values that are short where power is high
    # and long where power has decayed, ordered according to `direction`.
    # Total number of grid points across the whole fiber:
    # each span has StPS steps, plus one extra boundary point at the very end.
    model_steps = Nsp * StPS + 1
    cd_length = np.zeros(model_steps)   # dispersion (linear) step lengths
    nl_length = np.zeros(model_steps)   # nonlinear step lengths

    # Fill in the length arrays span-by-span.
    for NN in range(Nsp):          # loop over spans
        for MM in range(StPS):     # loop over steps within a span
            # Nonlinear step length: just reuse the per-span step pattern,
            # since each span starts fresh at full input power.
            nl_length[NN * StPS + MM] = step_size[MM]

            # Dispersion step length: symmetric SSFM merges the second half
            # of the previous nonlinear step with the first half of the
            # current one into a single "combined" linear-propagation step.
            # (MM + StPS - 1) % StPS wraps around to the LAST step of the
            # previous span/segment when MM == 0.
            cd_length[NN * StPS + MM] = (
                step_size[MM] / 2 + step_size[(MM + StPS - 1) % StPS] / 2
            )

    # The very first and very last dispersion steps have no neighboring
    # half-step to merge with (they're at the true boundaries of the whole
    # link), so they are just plain half-steps instead of combined ones.
    cd_length[0] = step_size[0] / 2
    cd_length[-1] = step_size[StPS - 1] / 2

    return cd_length, nl_length

In [ ]:
# Physical link parameters (all lengths in km, matching alpha_lin's units)
Lsp = 80    # Length of a single fiber span: 80 km (typical amplified-span length)
Nsp = 3     # Number of spans in the link (so total link length = 3 * 80 = 240 km)
StPS = 2    # Steps-per-span: split each span into 2 unequal (logarithmic) steps

# Call the function with all other args at their defaults:
#   alpha_lin=0.046  -> linear/Napierian attenuation in 1/k (this is the standard ~0.2 dB/km SMF loss converted to linear units: 0.2 / (10*log10(e)) ≈ 0.046 /km
# — so Lsp being in km here is exactly consistent with alpha_lin's units)   direction=-1 -> descending step order within each span (largest step first)
#   adjusting_factor=0.4 -> alpha_adj = 0.4 * 0.046 = 0.0184 /km used inside the log formula
cd_length, nl_length = calculate_step_sizes(Lsp, Nsp, StPS)
# At this point (still in km):
#   cd_length = [26.78, 40.0, 40.0, 40.0, 40.0, 40.0, 13.22]   (7 = Nsp*StPS+1 entries)
#   nl_length = [53.56, 26.44, 53.56, 26.44, 53.56, 26.44, 0.0]

# --- Unit conversion: km -> m ---
# The SSFM propagation loop (beta2, gamma, dz in the NLSE) almost always expects
# distances in meters, so every entry is scaled by 1e3.
# np.array(...) here is a no-op / defensive cast: the function already returns
# np.zeros(...) arrays, so this doesn't change anything — it just guards against
# a future version of calculate_step_sizes returning a plain Python list.
cd_length = 1e3 * np.array(cd_length)   # km → m as NumPy array
nl_length = 1e3 * np.array(nl_length)   # km → m as NumPy array

In [ ]:
"""
Learned Digital Back-Propagation (LDBP) model, deep-unfolded from SSFM.

Three pieces, chained in ComplexQAMModel:
  CircularConv1D   -> the LINEAR step  (inverts chromatic dispersion, D operator)
  KerrNonlinearity -> the NONLINEAR step (inverts Kerr self-phase modulation, N operator)
  ComplexQAMModel  -> stacks k of these [CD, NL] pairs, one pair per fiber segment,
                       mirroring how the physical fiber alternates dispersion and
                       nonlinearity along its length.
"""

import tensorflow as tf


class CircularConv1D(tf.keras.layers.Layer):
    """
    Trainable, symmetric (linear-phase), circularly-padded COMPLEX FIR filter.
    Represents the "D operator" of one SSFM/DBP step: compensating chromatic
    dispersion over a segment of fiber. Only half the taps are stored as
    weights (the CD impulse response satisfies h[-m] = h[m], since it comes
    from inverse-transforming H(w) = exp(j*beta2*dz*w^2/2), which is even in w);
    the other half is reconstructed by mirroring.
    """

    def __init__(self, filters=1, kernel_size=21, kernel_real_init=None, kernel_imag_init=None):
        super(CircularConv1D, self).__init__()

        # Odd kernel_size guarantees one unambiguous center tap (index 0 of the
        # half-kernel below) around which the mirror-symmetry trick works.
        if kernel_size % 2 == 0:
            raise ValueError("kernel_size must be odd to enforce symmetry")

        self.filters = filters
        self.kernel_size = kernel_size

        # For h[m] = h[-m], m = -(kernel_size//2) ... +(kernel_size//2), you only
        # need to store m = 0 ... kernel_size//2, i.e. (kernel_size//2)+1 taps.
        self.half_size = (kernel_size // 2) + 1

        # Taps from a physics-based filter design (e.g. least-squares CD filter)
        # used to INITIALIZE the trainable weights, instead of random init.
        self.kernel_real_init = kernel_real_init
        self.kernel_imag_init = kernel_imag_init

    def build(self, input_shape):
        # Trainable half-kernel, shape (half_size, in_channels=1, filters) ->
        # matches tf.nn.conv1d's expected [filter_width, in_channels, out_channels].
        self.kernel_real_half = self.add_weight(
            shape=(self.half_size, 1, self.filters),
            initializer=tf.constant_initializer(self.kernel_real_init),
            trainable=True,
            name="kernel_real_half"
        )

        self.kernel_imag_half = self.add_weight(
            shape=(self.half_size, 1, self.filters),
            initializer=tf.constant_initializer(self.kernel_imag_init),
            trainable=True,
            name="kernel_imag_half"
        )

    def call(self, inputs):
        # --- Rebuild the full symmetric kernel from the stored half ---
        # kernel_real_half = [h0, h1, ..., hM]  (h0 = center tap, hM = edge tap)
        # [-1:0:-1] takes indices M, M-1, ..., 1 (i.e. hM..h1), skipping h0.
        # concat -> [hM,...,h1, h0, h1,...,hM]  -- this is already a palindrome
        # (verified numerically), so the tf.reverse right after it is a no-op,
        # kept here only for defensive/explicit symmetry, not for correctness.
        kernel_real = tf.concat([self.kernel_real_half[-1:0:-1], self.kernel_real_half], axis=0)
        kernel_imag = tf.concat([self.kernel_imag_half[-1:0:-1], self.kernel_imag_half], axis=0)

        kernel_real = tf.reverse(kernel_real, axis=[0])  # no-op given the palindrome above
        kernel_imag = tf.reverse(kernel_imag, axis=[0])  # no-op given the palindrome above

        # Split the complex input into its real/imag channels
        # inputs: [batch, seq_len, 2], inputs[...,0]=Re(x), inputs[...,1]=Im(x)
        real_part, imag_part = inputs[..., 0], inputs[..., 1]

        #  Circular (periodic) padding, not zero-padding
        # Wrap the tail onto the front and the head onto the back, so the block
        # is treated as one period of a periodic signal (matches the periodicity
        # assumption used elsewhere in block-wise SSFM processing).
        padding_size = self.kernel_size // 2
        real_padded = tf.concat([real_part[:, -padding_size:], real_part, real_part[:, :padding_size]], axis=1)
        imag_padded = tf.concat([imag_part[:, -padding_size:], imag_part, imag_part[:, :padding_size]], axis=1)

        # tf.nn.conv1d needs an explicit channel dim: [batch, width, in_channels=1]
        real_padded = real_padded[..., tf.newaxis]
        imag_padded = imag_padded[..., tf.newaxis]

        # Complex convolution via 4 real convolutions
        # (a+jb)*(c+jd) = (ac-bd) + j(ad+bc); signal = re+j*im, kernel = kr+j*ki.
        # padding='VALID' is correct: we already circularly padded by
        # kernel_size//2 on each side, so output length == input length.
        real_conv = tf.nn.conv1d(real_padded, kernel_real, stride=1, padding='VALID') - \
                    tf.nn.conv1d(imag_padded, kernel_imag, stride=1, padding='VALID')
        imag_conv = tf.nn.conv1d(real_padded, kernel_imag, stride=1, padding='VALID') + \
                    tf.nn.conv1d(imag_padded, kernel_real, stride=1, padding='VALID')

        # Recombine into the same [batch, seq_len, 2] real/imag convention.
        # NOTE: real_conv/imag_conv already carry a channel dim of size `filters`
        # (from conv1d's output), so concat (merging along an EXISTING axis) is
        # the right op here -- contrast with KerrNonlinearity below, which uses
        # tf.stack because its real/imag parts have no channel dim yet.
        output = tf.concat([real_conv, imag_conv], axis=-1)
        return output


class KerrNonlinearity(tf.keras.layers.Layer):
    """
    Memoryless nonlinear phase-rotation layer -- the "N operator" of one
    SSFM/DBP step: compensating the Kerr effect (self-phase modulation) over
    a segment of fiber.

    Physical model:  A_out(t) = A_in(t) * exp( j * gamma * L_eff * |A_in(t)|^2 )

    - gamma is passed NEGATIVE here (e.g. -1.3): forward propagation accumulates
      a nonlinear phase of +gamma_phys*L_eff*|A|^2, so back-propagation applies
      the inverse rotation, i.e. the same operator with gamma flipped in sign.
    - L_eff ("effective length") accounts for the fact that optical power decays
      along the segment due to fiber loss (alpha), so nonlinear phase doesn't
      accumulate uniformly with distance -- it's power-weighted:
          L_eff = integral_0^L exp(-alpha*z) dz = (1 - exp(-alpha*L)) / alpha
      This is the standard effective-length formula from fiber nonlinear optics.

    Has NO trainable weights -- gamma, fiber_length (L), and alpha are all
    fixed constants, so only the CircularConv1D layers in the model actually
    learn anything; this layer stays pinned to its physics-derived value.
    """

    def __init__(self, gamma=-1.3, fiber_length=1.0, alpha=0.2):
        super(KerrNonlinearity, self).__init__()

        # Nonlinear coefficient (sign already flipped for inversion, see above).
        self.gamma = tf.convert_to_tensor(gamma, dtype=tf.float32)

        # Physical length of this segment, and fiber attenuation coefficient.
        self.fiber_length = tf.constant(fiber_length, dtype=tf.float32)
        self.alpha = tf.constant(alpha, dtype=tf.float32)

        # Effective nonlinear length, computed once at construction time.
        self.L_eff = (1 - tf.exp(-self.alpha * self.fiber_length)) / self.alpha

    def call(self, inputs):
        inputs = tf.cast(inputs, dtype=tf.float32)

        # Real/imag channels of the complex input, shape [batch, seq_len].
        real_part, imag_part = inputs[..., 0], inputs[..., 1]

        # Instantaneous optical power/intensity |A(t)|^2, sample by sample.
        intensity = real_part**2 + imag_part**2

        # Nonlinear phase phi_NL[n] = gamma * L_eff * |A[n]|^2, expressed as the
        # real/imag parts of the phasor exp(j*phi_NL) via Euler's formula.
        phase_shift_real = tf.math.cos(self.gamma * self.L_eff * intensity)   # = cos(phi_NL)
        phase_shift_imag = tf.math.sin(self.gamma * self.L_eff * intensity)   # = sin(phi_NL)

        # Complex multiply: A_out = A_in * exp(j*phi_NL)
        #   (a+jb)*(cos(phi)+j sin(phi)) = (a cos phi - b sin phi) + j(a sin phi + b cos phi)
        output_real = real_part * phase_shift_real - imag_part * phase_shift_imag
        output_imag = real_part * phase_shift_imag + imag_part * phase_shift_real

        # tf.stack (not concat) because real_part/imag_part have no channel dim
        # yet -- stack creates the new trailing axis of size 2.
        output = tf.stack([output_real, output_imag], axis=-1)
        return output


class ComplexQAMModel(tf.keras.Model):
    """
    The full deep-unfolded DBP receiver: a chain of [CircularConv1D,
    KerrNonlinearity] pairs, one pair per fiber segment, applied in sequence.

    This directly mirrors the split-step structure of the physical channel:
    forward propagation alternates dispersion and nonlinearity in small steps
    because the two operators don't commute; this model applies the same
    alternating structure in reverse, once per segment, to undo it.
    """

    def __init__(self, kernel_size, cd_length, nl_length, cdfilter):
        super(ComplexQAMModel, self).__init__()

        # Number of unfolded stages = number of fiber segments being compensated.
        self.num_blocks = len(cd_length)
        self.blocks = []

        for i in range(self.num_blocks):
            # Physics-based CD filter taps for THIS segment's length, from your
            # least-squares/Fresnel-integral filter design (cdfilter.get_filter),
            # used to initialize (not replace) the trainable CircularConv1D taps.
            weightr, weighti = cdfilter.get_filter(cd_length[i], kernel_size)

            self.blocks.append([
                # Linear step for this segment: trainable, physics-initialized.
                CircularConv1D(filters=1, kernel_size=kernel_size,
                                kernel_real_init=weightr, kernel_imag_init=weighti),
                # Nonlinear step for this segment: fixed, not trainable.
                # nl_length[i] sets this segment's L_eff (gamma, alpha shared
                # across all blocks here).
                KerrNonlinearity(gamma=-1.3, fiber_length=nl_length[i], alpha=0.2),
            ])

    def call(self, inputs):
        x = inputs
        # Apply CD_1 -> NL_1 -> CD_2 -> NL_2 -> ... -> CD_k -> NL_k in sequence.
        for conv_layer, kerr_layer in self.blocks:
            x = conv_layer(x)   # undo dispersion for this segment
            x = kerr_layer(x)   # undo Kerr nonlinearity for this segment
        return x


# Model parameters
# NOTE: none of these three module-level variables are actually threaded into
# ComplexQAMModel below -- filters and gamma are hardcoded inside the class
# (filters=1 in CircularConv1D, gamma=-1.3 in KerrNonlinearity), so changing
# `filters` or `gamma` here would silently have NO effect on the model. Only
# `kernel_size` is actually passed through and used.
filters = 2          # Unused by ComplexQAMModel as written (see note above)
kernel_size = 21      # Kernel size for circular convolution -- this one IS used
gamma = -1.3          # Unused by ComplexQAMModel as written (see note above)

# cd_length, nl_length, cdfilter must be defined elsewhere (e.g. per-segment
# fiber lengths and a filter-design helper object) before this line runs.
model = ComplexQAMModel(kernel_size, cd_length, nl_length, cdfilter)

In [ ]:


"""
Data pipeline for training the LDBP model.

Loads the complex received signal (data) and the complex transmitted/target
signal (label), chops both into fixed-length blocks matching the model's
[batch, seq_len, 2] real/imag input convention, splits into train/test, and
wraps everything in an infinite shuffled-minibatch generator for a custom
(GradientTape-style) training loop.
"""

import numpy as np

# Load raw complex-valued arrays
data = np.load(data_path)    # received/distorted signal -> model INPUT
# data = data/np.max(np.abs(data))     # optional peak normalization (disabled)
label = np.load(label_path)  # transmitted/clean signal  -> model TARGET
# label = label/np.max(np.abs(label))  # optional peak normalization (disabled)


# Config
split = 0.8        # fraction of samples used for training (80/20 train/test)
batch_size = 32     # minibatch size
NTaps = 400          # samples per training block (the model's seq_len).
                     # Chosen large relative to the per-stage FIR kernel_size=21
                     # so that the circular-padding "wrap-around" approximation
                     # in CircularConv1D only disturbs a small fraction of each
                     # block, and large enough to comfortably span the total CD
                     # memory being compensated across the link.

# Reassign `split` from a fraction (0.8) to an absolute sample INDEX -- this
# is the index in the raw, un-chunked array where train data ends and test
# data begins.
split = int(split * len(data))


def data_adjuster(data, ntaps=NTaps):
    """
    Reshape a flat complex 1D array into (num_blocks, ntaps, 2):
      - truncate the tail so length is an exact multiple of ntaps (drops the
        leftover partial block, if any)
      - reshape into (num_blocks, ntaps) blocks
      - split each complex block into real/imag and stack on a new last axis
        -> matches the model's expected [batch-ish, seq_len, 2] input format
    """
    if data.shape[0] % ntaps != 0:
        data = data[:(data.shape[0] // ntaps) * ntaps]   # drop incomplete tail block
    data = data.reshape(-1, ntaps)                        # (num_blocks, ntaps)
    return np.stack([np.real(data), np.imag(data)], axis=2)  # (num_blocks, ntaps, 2)


def label_adjuster(label, ntaps=NTaps):
    """Same block-and-split logic as data_adjuster, applied to the target signal."""
    if label.shape[0] % ntaps != 0:
        label = label[:(label.shape[0] // ntaps) * ntaps]
    label = label.reshape(-1, ntaps)
    return np.stack([np.real(label), np.imag(label)], axis=2)


# Train/test split, done BEFORE chunking into blocks
# Splitting the raw (flat) array first, then chunking each half independently,
# guarantees no single NTaps-length block ever straddles the train/test
# boundary -- each half starts its own fresh block grid at index 0. This
# avoids leaking a few samples of "future" test data into a training block
# (or vice versa) at the seam.
train_data = data_adjuster(data[:split])
test_data = data_adjuster(data[split:])
train_label = label_adjuster(label[:split])
test_label = label_adjuster(label[split:])


# Infinite shuffled minibatch generator
def data_iterator(data, label, batch_size):
    """
    Yields (data_batch, label_batch) pairs forever, re-shuffling the batch
    order every epoch.

    NOTE: this shuffles the ORDER of pre-formed, fixed contiguous batches of
    `batch_size` blocks -- not a full shuffle of individual blocks across
    batch boundaries. Cheaper, but the composition of each batch (which
    blocks are grouped together) never changes across epochs, only which
    batch comes first/second/etc.

    NOTE: np.random.seed(123) reseeds the GLOBAL numpy RNG, and this function
    is called twice (once for train_feeder, once for test_feeder). Because
    generators are lazy, the seed(123) call only runs on each generator's
    FIRST `next()`. If both generators are advanced in an interleaved
    fashion (e.g. training then periodically evaluating), the test feeder's
    first call will reseed the shared global RNG, which can disturb the
    intended reproducibility of the train feeder's ongoing shuffle sequence.
    Using two independent `np.random.default_rng(seed)` generator objects
    instead would avoid this cross-talk.
    """
    nums = np.arange(len(label) // batch_size)  # one entry per full batch
                                                 # (any leftover < batch_size
                                                 # samples at the end are
                                                 # never yielded, every epoch)
    np.random.seed(123)
    while True:
        np.random.shuffle(nums)          # new random batch order each epoch
        for i in nums:
            yield data[i*batch_size:(i+1)*batch_size], label[i*batch_size:(i+1)*batch_size]


train_feeder = data_iterator(train_data, train_label, batch_size)
test_feeder = data_iterator(test_data, test_label, batch_size)

In [ ]:
def myfirFilter(signal):
    """
    Applies a matched/pulse-shaping FIR filter (via convolution) to a 1D
    real-valued signal tensor, using an RRC (root-raised-cosine) pulse.
    Input:  signal -> tensor of shape (batch_size, sequence_length)
    Output: filtered signal -> tensor of shape (batch_size, sequence_length, 1)
    """
    # Build the pulse-shaping filter
    param = parameters()
    param.pulseType   = 'rrc'    # Root-raised-cosine pulse (options: rect, nrz, rrc, rc, doubinary)
    param.SpS         = 2        # Samples per symbol (oversampling factor)
    param.nFilterTaps = 41       # Number of FIR filter taps (filter length)
    param.rollOff     = 0.01     # RRC roll-off factor (controls excess bandwidth)

    pulse = pulseShape(param)     # Generate the RRC pulse shape as a numpy array

    # Normalize the pulse so its peak amplitude is 1 (avoids divide-by-zero with `or 1.0`)
    denom = np.max(np.abs(pulse)) or 1.0
    pulse = pulse / denom
    pulse = tf.cast(pulse, tf.float32)   # Convert pulse to a TF float32 tensor

    # Prepare the input signal for 1D convolution
    signal = tf.cast(signal, tf.float32)
    signal = tf.expand_dims(signal, axis=-1)   # Add a "channel" dimension: (batch, seq_len, 1)

    # Grab dynamic shape info (needed later to crop back to original length)
    batch_size, original_length, channels = tf.shape(signal)[0], tf.shape(signal)[1], tf.shape(signal)[2]

    # Reshape the pulse into conv1d's expected kernel shape: (filter_width, in_channels, out_channels)
    pulse = tf.reshape(pulse, [-1, 1, 1])

    # Prepend a single zero sample to the signal (effectively a 1-sample delay/shift trick,
    # likely to align the filter's group delay or avoid an off-by-one boundary effect)
    padded_signal = tf.concat([tf.zeros([batch_size, 1, channels], dtype=tf.float32), signal], axis=1)

    # Convolve signal with the pulse shape (this is the actual FIR filtering step)
    filtered_signal = tf.nn.conv1d(padded_signal, pulse, stride=1, padding='SAME')

    # Trim the output back down to the original sequence length
    # (compensates for the extra zero-padding sample added above)
    shifted_filtered_signal = filtered_signal[:, :original_length, :]

    return shifted_filtered_signal


def dsp(data):
    """
    Runs the FIR pulse-shaping filter independently on the real and
    imaginary parts of a complex-valued signal (stored as 2 real channels),
    then restacks them.
    Input:  data -> shape (batch_size, sequence_length, 2)  [last dim = (I, Q)]
    Output: filtered_signal -> shape (batch_size, sequence_length, 2)
    """
    # Split I/Q (real/imaginary) components
    real_part = data[..., 0]  # (batch_size, sequence_length)
    imag_part = data[..., 1]  # (batch_size, sequence_length)

    # Filter each component separately with the same FIR filter
    filtered_real = myfirFilter(real_part)
    filtered_imag = myfirFilter(imag_part)

    # Drop the extra channel dim added inside myfirFilter
    filtered_real = filtered_real[..., 0]
    filtered_imag = filtered_imag[..., 0]

    # Recombine I/Q into a single tensor again
    filtered_signal = tf.stack([filtered_real, filtered_imag], axis=-1)  # (batch, seq_len, 2)

    return filtered_signal


def custom_loss(y_true, y_pred, kernel_size=kernel_size):
    """
    Custom training loss for a signal-processing / equalization model.
    Compares predicted vs. true signals after:
      1) matched filtering (dsp)
      2) downsampling to symbol rate (::2)
      3) removing filter-transient (edge) samples
      4) correcting for a common-phase-error (CPE) rotation
      5) measuring mean squared error in the complex plane
    """
    skip = int(kernel_size // 2)   # Number of edge samples to discard (transient from filtering)

    # Process predictions
    y_pred = dsp(y_pred)  # matched filter the predicted I/Q signal -> (batch, seq_len, 2)
    # Downsample by 2 (::2) to go from sample-rate back to symbol-rate, and pack into complex numbers
    y_pred_complex = tf.complex(y_pred[:, ::2, 0], y_pred[:, ::2, 1])

    # (Commented out in original: per-sample amplitude normalization of predictions)
    # max_abs_y_pred = tf.reduce_max(tf.abs(y_pred_complex), axis=-1, keepdims=True)
    # max_abs_y_pred_complex = tf.cast(max_abs_y_pred, dtype=tf.complex64)
    # y_pred_normalized = y_pred_complex / (max_abs_y_pred_complex + 1e-7)
    y_pred_normalized = y_pred_complex   # normalization currently skipped

    # --- Process ground truth the same way ---
    y_true = dsp(y_true)
    y_true_complex = tf.complex(y_true[:, ::2, 0], y_true[:, ::2, 1])

    # Cut off the filter's transient (warm-up/edge) region from both signals
    y_true_complex = y_true_complex[:, skip:]
    y_pred_normalized = y_pred_normalized[:, skip:]

    # --- Common Phase Error (CPE) correction ---
    # Correlate true & predicted symbols to estimate the best-fit constant phase offset
    tmp = tf.reduce_sum(tf.math.conj(y_true_complex) * y_pred_normalized, axis=-1, keepdims=True)
    phi_cpe = -tf.atan2(tf.math.imag(tmp), tf.math.real(tmp))   # phase angle of that correlation
    x_hat = y_pred_normalized * tf.exp(tf.complex(0.0, phi_cpe))  # rotate prediction to align phase

    # --- Compute error ---
    squared_diff = tf.square(tf.abs(y_true_complex - x_hat))   # squared magnitude of complex error

    # Average over symbols, then over the batch -> final scalar loss (this is essentially MSE / EVM^2)
    loss = tf.reduce_mean(squared_diff, axis=-1)
    return tf.reduce_mean(loss)


def mynorm(arr):
    """
    Normalizes a complex tensor by its per-sequence maximum magnitude,
    so all values lie within the unit circle (peak amplitude = 1).
    """
    max_abs_y_pred = tf.reduce_max(tf.abs(arr), axis=-1, keepdims=True)
    max_abs_y_pred_complex = tf.cast(max_abs_y_pred, dtype=tf.complex64)
    y_pred_normalized = arr / (max_abs_y_pred_complex + 1e-7)  # avoid divide-by-zero
    return y_pred_normalized


def snr(y_true, y_pred, kernel_size=kernel_size):
    """
    Computes a Signal-to-Noise-Ratio-like metric (inverse of MSE) between
    the true and predicted signals, after the same filtering/alignment
    pipeline as custom_loss, but with explicit amplitude normalization
    added via mynorm() before comparison.
    """
    skip = int(kernel_size // 2)

    # Matched-filter, downsample, and complexify the prediction
    y_pred = dsp(y_pred)
    y_pred_complex = tf.complex(y_pred[:, ::2, 0], y_pred[:, ::2, 1])

    # (Same commented-out normalization block as in custom_loss)
    # max_abs_y_pred = tf.reduce_max(tf.abs(y_pred_complex), axis=-1, keepdims=True)
    # max_abs_y_pred_complex = tf.cast(max_abs_y_pred, dtype=tf.complex64)
    # y_pred_normalized = y_pred_complex / (max_abs_y_pred_complex + 1e-7)
    y_pred_normalized = y_pred_complex

    # Matched-filter, downsample, and complexify the ground truth
    y_true = dsp(y_true)
    y_true_complex = tf.complex(y_true[:, ::2, 0], y_true[:, ::2, 1])

    # Trim transient edges
    y_true_complex = y_true_complex[:, skip:]
    y_pred_normalized = y_pred_normalized[:, skip:]

    # Explicitly normalize both signals to unit peak amplitude before comparing
    y_true_complex = mynorm(y_true_complex)
    y_pred_normalized = mynorm(y_pred_normalized)

    # Common-phase-error correction (same as custom_loss)
    tmp = tf.reduce_sum(tf.math.conj(y_true_complex) * y_pred_normalized, axis=-1, keepdims=True)
    phi_cpe = -tf.atan2(tf.math.imag(tmp), tf.math.real(tmp))
    x_hat = y_pred_normalized * tf.exp(tf.complex(0.0, phi_cpe))

    # Mean squared error between aligned true and predicted signals
    squared_diff = tf.square(tf.abs(y_true_complex - x_hat))
    loss = tf.reduce_mean(squared_diff, axis=-1)

    # Return the reciprocal of the MSE -> larger value = higher "SNR" (less error)
    return 1 / tf.reduce_mean(loss)

In [ ]:
"""
Model compilation: set up optimizer, loss, and metrics for training.
"""

import tensorflow as tf


model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),loss=custom_loss,metrics=[snr])

 # --- Optimizer ---
    # Adam with a small learning rate (1e-5 = 0.00001)
    # Adam is adaptive: per-parameter learning rates that adjust based on gradient
    # history. Better than vanilla SGD for complex, non-convex losses like this.
    # Small LR is deliberate: the model starts physics-initialized (CD filters
    # designed via Fresnel integrals), so we want small, careful updates rather
    # than aggressive jumps. Lets training fine-tune the design without overshooting.
    # --- Loss function ---
    # custom_loss: the function defined earlier that computes symbol-level MSE
    # after DSP preprocessing (matched filter, downsample, phase correction).
    # This is what gradient descent will minimize during training.
    # NOT tf.keras.losses.MeanSquaredError or built-in losses — custom_loss
    # is your domain-specific function that accounts for receiver DSP.
    # --- Metrics ---
    # snr: the function defined earlier that returns 1/MSE (a human-readable
    # SNR-like metric). Keras will compute this after every batch/epoch and
    # log it, but it is NOT used for training — only for monitoring/reporting.
    # Higher SNR = lower error = model is learning. Easier to read than raw MSE.


# model.summary()
# would print the layer structure and parameter counts. Useful for debugging, but not needed once you've verified the architecture.

In [ ]:

"""
Training callbacks: custom logging and adaptive learning rate scheduling.
Callbacks are hooks that Keras runs at specified points during training
(start/end of epoch, batch, etc.) to monitor, log, or dynamically adjust
training parameters.
"""

import os
import tensorflow as tf

class LogToFile(tf.keras.callbacks.Callback):
    """
    Custom callback that writes training metrics to a file after every epoch.
    Keras callbacks inherit from tf.keras.callbacks.Callback and override
    hook methods like on_epoch_end(), on_batch_end(), etc. This one logs
    loss and SNR to both stdout (print) and a file (for later analysis).
    """

    def __init__(self, log_file='training_logs.txt'):
        """
        Args:
            log_file (str): path where logs will be appended each epoch
        """
        super(LogToFile, self).__init__()
        self.log_file = log_file

    def on_epoch_end(self, epoch, logs=None):
        """
        Called by Keras after each epoch completes.

        Args:
            epoch (int): epoch number (0-indexed)
            logs (dict): dictionary of metrics computed that epoch
                        e.g. {'loss': 0.0234, 'snr': 42.5}
        """
        logs = logs or {}  # handle None if no metrics are available

        # Open the log file in append mode ('a') and write one line per epoch
        with open(self.log_file, 'a') as f:
            # Build a human-readable string: "Epoch 1: loss: 0.02345678, snr: 42.12345678"
            log_str = f"Epoch {epoch+1}: "
            log_str += ", ".join([f"{key}: {value:.8f}" for key, value in logs.items()])

            # Write to file
            f.write(log_str + "\n")
            # Also print to stdout for real-time monitoring
            print(log_str)

lr_plateau = tf.keras.callbacks.ReduceLROnPlateau(    monitor="loss", factor=0.5,  patience=10, verbose=2,   mode="min", min_lr=1e-7)
    # Monitor this metric. If it stops improving, reduce the learning rate.
    # "loss" = monitor training loss (the value returned by custom_loss each epoch)
    # Multiply learning rate by this factor when loss plateaus.
    # factor=0.5 means: if LR was 1e-5, new LR becomes 0.5 * 1e-5 = 5e-6
    # How many epochs with no improvement before triggering LR reduction.
    # patience=10 means: if loss doesn't decrease for 10 consecutive epochs,
    # reduce LR. Then reset the counter; if improvement resumes, no further
    # reduction. If another 10 epochs pass without improvement, reduce again.
    # Verbosity level. verbose=2 prints a message when LR is reduced.
    # (verbose=0 = silent, verbose=1 = progress bar, verbose=2 = detailed)
    # "min" = we're trying to minimize the monitored metric (loss).
    # (use "max" if monitoring a metric like accuracy where higher is better)
    # Lower bound on the learning rate. Even if ReduceLROnPlateau wants to
    # reduce further, it won't go below 1e-7. Prevents LR from becoming
    # so small that training essentially freezes.

# Path to Google Drive (assumes you're running in Google Colab).
# In a local environment, change this to a local directory path.
log_dir = "/content/drive/MyDrive/MyNewProjectFolder"

# Create the directory if it doesn't already exist.
# exist_ok=True prevents an error if the directory is already there.
# This is where all logs and potentially checkpoints will be saved.
os.makedirs(log_dir, exist_ok=True)

# Build the full path to the training log file.
log_path = os.path.join(log_dir, "training_logs.txt")

# Instantiate the custom logging callback with the log file path.
# This will be passed to model.fit() later.
log_callback = LogToFile(log_file=log_path)


In [ ]:
"""
Model training: fit the LDBP receiver to minimize symbol-level error.
model.fit() is Keras's high-level training API. It runs the full training
loop: forward pass → loss → backprop → weight update, repeating for N epochs.
"""

import os
import tensorflow as tf

history = model.fit(train_feeder,epochs=200, verbose=2, steps_per_epoch=train_label.shape[0] // batch_size, callbacks=[log_callback],
                    validation_data=test_feeder,validation_steps=test_label.shape[0] // batch_size)

    # --- Input data ---
    # Generator yielding (batch_x, batch_y) tuples infinitely.
    # Each call to next(train_feeder) returns a fresh minibatch of size 32.
    # --- Number of epochs ---
    # An epoch is one pass through all training data (all complete minibatches).
    # 200 epochs = train for 200 × (len(train_label) // batch_size) steps.
    # --- Verbosity ---
    # verbose=0: silent (no output)
    # verbose=1: progress bar (one line per epoch with bar animation)
    # verbose=2: one line per epoch with detailed metrics (loss, SNR, val_loss, val_SNR)
    # --- Steps per epoch ---
    # How many minibatches to process before declaring one epoch complete.
    # train_label.shape[0] = number of training blocks (e.g. 1000 blocks)
    # batch_size = 32
    # steps_per_epoch = 1000 // 32 = 31 steps per epoch (31 × 32 = 992 samples,
    # 8 leftover samples discarded per epoch — data_iterator only yields
    # complete batches).

    # --- Callbacks ---
    # List of callbacks Keras will invoke during training:
    # [log_callback] means LogToFile fires after each epoch, writing metrics
    # to training_logs.txt.
    # You can also pass [log_callback, lr_plateau] if you want both the custom
    # logging AND adaptive learning rate scheduling.

    # --- Validation data ---
    # Generator yielding (test_batch_x, test_batch_y) tuples infinitely.
    # Each epoch, after training on all steps_per_epoch training batches,
    # Keras runs the model on validation_data and computes validation loss
    # and validation metrics (logged as 'val_loss', 'val_snr').

    # --- Validation steps ---
    # How many minibatches from test_feeder to process for validation
    # (not used for training, only to evaluate generalization).
    # test_label.shape[0] // batch_size = number of complete test batches.

# model.save(os.path.join(name, 'model.h5'))

# Saves the trained model to disk as a single HDF5 file (.h5 format).
# Includes:
#   - Model architecture (layer structure, weights, biases)
#   - All trained weights (updated by SGD over 200 epochs)
#   - Custom objects (custom_loss, snr functions if registered)
#
# HDF5 format is legacy; modern Keras uses SavedModel format instead:
#   model.save(os.path.join(name, 'model'))  # saves as SavedModel directory
#
# Later, you can load and use the saved model:
#   loaded_model = tf.keras.models.load_model(model_path,
#       custom_objects={'custom_loss': custom_loss, 'snr': snr})
#   predictions = loaded_model.predict(test_data)
#
# Why save?
#   - Persist weights across sessions (survive Colab disconnect)
#   - Deploy to production
#   - Evaluate on new test data later without retraining
#   - Share trained model with colleagues




In [ ]:
"""
Inference pipeline: use trained model to predict on test data and measure
symbol-level performance.

This mirrors the DSP preprocessing from custom_loss(), but applied post-training
to evaluate the model's actual performance on never-before-seen test blocks.
"""

import numpy as np
import tensorflow as tf

# Take the first `batch_size` blocks from the test set (unseen during training).
# batch_x = received (distorted) signal
# batch_y = transmitted (clean) signal
batch_x = test_data[:batch_size]
batch_y = test_label[:batch_size]

print("Input shape:", batch_x.shape)
#   32 blocks (batch), 400 samples per block (2 sps), 2 channels (Re, Im)

# model.predict() = forward pass only (no backprop). Returns numpy array
# (not TF tensor). Faster than model() because it skips gradient tracking.
# y_pred.shape = (32, 400, 2), same as input
#   Still oversampled (2 sps), still [Re, Im] format
#   But now the signal has been processed through all k stages of LDBP:
#   CircularConv1D (CD compensation) + KerrNonlinearity (nonlinear compensation)
#   repeated for each segment of the fiber channel.
y_pred = model.predict(batch_x)

# dsp() function (defined in loss) requires TF tensors, not numpy arrays.
# Cast to float32 to match the model's internal dtype.
y_pred_tf = tf.convert_to_tensor(y_pred, dtype=tf.float32)
y_true_tf = tf.convert_to_tensor(batch_y, dtype=tf.float32)

# CRITICAL: use the SAME dsp() function (RRC pulse shaping + filtering) that
# was applied during training loss. This ensures fair comparison: both the model
# and ground truth are filtered identically.
#
# dsp(data):
#   - Extracts Re and Im parts
#   - Applies myfirFilter (RRC matched filter) to each independently
# After filtering, the received signal is "shaped up" at the symbol rate.
y_pred_dsp = dsp(y_pred_tf)
y_true_dsp = dsp(y_true_tf)

# Input was oversampled at 2 samples per symbol (2 sps).
# After matched filtering, keep every 2nd sample: move from 2 sps -> 1 sps.
#
# y_pred_dsp[:,::2,0] = every 2nd sample of the real part
# This gives one sample per symbol (symbol-rate timing).
#
# tf.complex() packs Re and Im into a true complex-valued tensor.
# Output shape: (32, 200) where 200 = 400 / 2 (one sample per symbol)
y_pred_sym = tf.complex(y_pred_dsp[:,::2,0], y_pred_dsp[:,::2,1])
y_true_sym = tf.complex(y_true_dsp[:,::2,0], y_true_dsp[:,::2,1])


# The first kernel_size//2 symbols of each block are distorted by:
#   - Circular convolution boundary effects (CircularConv1D wraps the edges)
#   - Matched filter startup transient (RRC filter hasn't fully settled)
# Skip these unreliable symbols when computing error metrics.
# skip = 21 // 2 = 10 symbols
# y_pred_sym[:, skip:] discards first 10 symbols, keeps symbols 10 to 199.

skip = kernel_size // 2
y_pred_sym = y_pred_sym[:, skip:]
y_true_sym = y_true_sym[:, skip:]


# TensorFlow tensors can't be easily plotted or stored; convert to numpy
# for downstream analysis (plotting, BER computation, etc.).
y_pred_sym = y_pred_sym.numpy()
y_true_sym = y_true_sym.numpy()


# Reshape from (32 blocks, 190 symbols/block) -> (32*190,) = (6080,)
# One long sequence of recovered symbols vs. true transmitted symbols.
# Makes it easy to plot on a constellation diagram, compute error distribution, etc.
pred = y_pred_sym.reshape(-1)  # shape: (6080,)
true = y_true_sym.reshape(-1)  # shape: (6080,)


# After LDBP, there's an arbitrary phase offset between pred and true.
# This is NOT an error in the LDBP model — it's a residual carrier-phase
# error that a separate carrier-phase recovery (CPE) stage would handle.

# To measure the LDBP stage's performance fairly, remove this ambiguous
# phase offset before computing symbol error rate (SER).

# Math: find the phase φ that minimizes ||true - pred*e^(jφ)||^2
# Closed form: φ = -angle(sum(conj(true) * pred))
#   This is the inner product of true and pred in the complex plane;
#   its angle tells us the best-fit rotation.
#
# Example:
#   sum(conj(true)*pred) = 100 + 50j  (magnitude ~111, angle ~27°)
#   phi = -atan2(50, 100) = -27°
#   Apply inverse rotation: pred_corrected = pred * exp(j*27°)
tmp = np.sum(np.conj(true) * pred)          # complex inner product
phi = -np.arctan2(np.imag(tmp), np.real(tmp))  # phase to rotate by

# Apply phase correction to predictions
pred = pred * np.exp(1j * phi)

print("Final symbol shape:", pred.shape)




In [ ]:
print(pred[:5])
print(true[:5])

In [ ]:
print("Mean Abs Error:", np.mean(np.abs(pred - true)))

In [ ]:
# plot constellation daigram after ML between true value and predicted value  and in x axis In-pahse value  and in y axis quadrature value
import matplotlib.pyplot as plt

# (Optional normalization for fair comparison)
pred_norm = pred / (np.sqrt(np.mean(np.abs(pred)**2)) + 1e-8)
true_norm = true / (np.sqrt(np.mean(np.abs(true)**2)) + 1e-8)

plt.figure(figsize=(6,6))

# Ground truth
plt.scatter(true_norm.real, true_norm.imag,
            color='blue', alpha=0.5, label='Ground Truth')

# Model output
plt.scatter(pred_norm.real, pred_norm.imag,
            color='red', alpha=0.5, label='Predicted')

plt.axhline(0)
plt.axvline(0)

plt.xlabel("In-Phase (I)")
plt.ylabel("Quadrature (Q)")
plt.title("Constellation Diagram (After ML)")
plt.legend()
plt.grid()
plt.axis('equal')
plt.show()

In [ ]:
# Normalize the predicted signal to unit average power (RMS normalization),
# np.abs(pred)**2 -> instantaneous power of each complex sample
# np.mean(...)    -> average power over the signal
# np.sqrt(...)    -> RMS (root-mean-square) amplitude
# Dividing by this scales the signal so its average power becomes 1.
# The 1e-8 avoids division by zero if the signal is all zeros.
pred_n = pred / (np.sqrt(np.mean(np.abs(pred)**2)) + 1e-8)

# Same RMS/power normalization applied to the ground-truth signal,
# so both pred_n and true_n are on the same average-power scale
true_n = true / (np.sqrt(np.mean(np.abs(true)**2)) + 1e-8)

In [ ]:
# The 4 amplitude levels used on each axis (I and Q) of a 16-QAM constellation.
# 16-QAM = 4 levels on the real axis x 4 levels on the imaginary axis = 16 total symbols.
levels = np.array([-3, -1, 1, 3])

def quantize(v):
    # "Slice" a single real-valued sample to the nearest valid constellation level.
    # np.abs(levels - v)   -> distance from v to each of the 4 allowed levels
    # np.argmin(...)       -> index of the closest level
    # levels[...]          -> snap v to that closest level
    return levels[np.argmin(np.abs(levels - v))]

def slicer_16qam(x):
    # Apply the per-axis quantizer independently to the real (I) and
    # imaginary (Q) parts of a complex-valued signal, since 16-QAM is
    # just two independent 4-level PAM signals combined into I and Q.
    real = np.array([quantize(r) for r in x.real])
    imag = np.array([quantize(i) for i in x.imag])

    # Recombine the quantized I and Q components into complex symbols,
    # i.e. the "decided" (hard-decision) constellation points.
    return real + 1j * imag

# Convert the continuous-valued, noisy predicted/true signals into
# discrete, ideal 16-QAM symbols by snapping each sample to its
# nearest valid constellation point.
pred_dec = slicer_16qam(pred_n)
true_dec = slicer_16qam(true_n)

In [ ]:
# Compare each decided prediction symbol to its corresponding true symbol.
# Complex equality checks both real (I) and imaginary (Q) parts at once,
# so a mismatch in either component counts as a symbol error.
# Result: boolean array -> True = wrong symbol decided, False = correct symbol.
errors = pred_dec != true_dec

# np.mean() on a boolean array treats True=1, False=0, so this computes:
#   (count of symbol errors) / (total number of symbols)
ser = np.mean(pred_dec != true_dec)
print("SER:", ser)

In [ ]:
# Gray-coded bit mapping for each 16-QAM amplitude level (per axis).
# Gray coding ensures that adjacent constellation levels differ by only
# 1 bit, so a small noise-induced slip to a neighboring level causes
# only a single bit error rather than multiple bit errors.
mapping = {
    -3: [0, 0],
    -1: [0, 1],
     1: [1, 1],
     3: [1, 0]
}

def symbols_to_bits(x):
    # Convert an array of decided complex 16-QAM symbols into a flat
    # bitstream, using the Gray-coded mapping above.
    bits = []
    for val in x:
        # Map the real (I) component's level to its 2 bits, append them
        bits.extend(mapping[np.real(val)])
        # Map the imaginary (Q) component's level to its 2 bits, append them
        # -> each 16-QAM symbol contributes 2 (I) + 2 (Q) = 4 bits total
        bits.extend(mapping[np.imag(val)])
    return np.array(bits)

# Convert the true (reference) decided symbols into their bit sequence
bits_true = symbols_to_bits(true_dec)

# Convert the predicted (equalizer/model output) decided symbols into their bit sequence
bits_pred = symbols_to_bits(pred_dec)

In [ ]:
#  calculate the average BER and print its value
ber = np.mean(bits_true != bits_pred)
print("BER:", ber)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf


# 1) SYMBOL EXTRACTION (use best global offset)

def extract_symbols(y_pred_dsp, y_true_dsp, SpS, skip):
    """
    After matched filtering (dsp), the signal is still at sample-rate
    (SpS samples per symbol). To recover one value per transmitted symbol,
    you must downsample by picking every SpS-th sample — but WHICH sample
    within each symbol period (the "timing offset") gives the best-aligned,
    lowest-error symbols isn't known in advance. This function tries every
    possible offset (0 to SpS-1) and keeps whichever gives the smallest error,
    """
    best_offset = 0
    best_err = 1e9   # start with a very large "worst case" error

    for offset in range(SpS):
        # Downsample prediction and ground truth starting at this offset,
        # taking every SpS-th sample, and pack I/Q into complex numbers
        yh = tf.complex(y_pred_dsp[:, offset::SpS, 0], y_pred_dsp[:, offset::SpS, 1])
        yt = tf.complex(y_true_dsp[:, offset::SpS, 0], y_true_dsp[:, offset::SpS, 1])

        # Drop the filter's transient/edge symbols, then flatten batch+seq into 1D
        yh = yh[:, skip:].numpy().reshape(-1)
        yt = yt[:, skip:].numpy().reshape(-1)

        # --- Common Phase Error (CPE) correction ---
        # Estimate the constant phase rotation between prediction and truth,
        # and rotate the prediction to align with the truth before measuring error
        tmp = np.sum(np.conj(yt) * yh)
        phi = -np.arctan2(np.imag(tmp), np.real(tmp))
        yh = yh * np.exp(1j * phi)

        # Mean absolute error between phase-aligned prediction and truth
        err = np.mean(np.abs(yh - yt))

        # Keep track of whichever offset gives the lowest error so far
        if err < best_err:
            best_err = err
            best_offset = offset
            pred_best = yh
            true_best = yt

    return pred_best, true_best, best_offset

# Run symbol extraction on the ML model's output (post-equalization)
pred_all, true_all, best_offset = extract_symbols(y_pred_dsp, y_true_dsp, SpS, skip)
print("Best offset:", best_offset)


# 2) BEFORE ML (for comparison)

# Matched-filter the RAW input (batch_x), i.e. the signal BEFORE the neural
# network touched it, so we can compare "before vs after ML" performance
y_in = dsp(tf.convert_to_tensor(batch_x, dtype=tf.float32))
y_in = y_in.numpy()

# Downsample using the SAME best_offset found above, for a fair/consistent comparison
y_in_sym = tf.complex(y_in[:, best_offset::SpS, 0], y_in[:, best_offset::SpS, 1])
y_in_sym = y_in_sym[:, skip:].numpy().reshape(-1)   # drop transient, flatten

# Phase-align the raw input symbols to the ground truth as well,
# so the "before" constellation isn't just rotated arbitrarily
tmp = np.sum(np.conj(true_all) * y_in_sym)
phi = -np.arctan2(np.imag(tmp), np.real(tmp))
y_in_sym = y_in_sym * np.exp(1j * phi)



# 3) NORMALIZATION

def norm(x):
    # RMS (average power) normalization -> scales x so its mean power is 1.
    # Puts pred/true/raw-input all on the same amplitude scale for fair comparison.
    return x / (np.sqrt(np.mean(np.abs(x)**2)) + 1e-8)

pred_all = norm(pred_all)
true_all = norm(true_all)
y_in_sym = norm(y_in_sym)



# 4) CONSTELLATION PLOTS

# Visualize the I/Q scatter ("constellation diagram") at 3 stages:
# raw input, ML-equalized output, and the ideal ground truth.
# Tighter, more distinct clusters = better signal quality / lower noise.
plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.scatter(y_in_sym.real, y_in_sym.imag, s=5, alpha=0.3)
plt.title("Before ML")
plt.axis('equal'); plt.grid()

plt.subplot(1, 3, 2)
plt.scatter(pred_all.real, pred_all.imag, s=5, alpha=0.3)
plt.title("After ML")
plt.axis('equal'); plt.grid()

plt.subplot(1, 3, 3)
plt.scatter(true_all.real, true_all.imag, s=5, alpha=0.3)
plt.title("Ground Truth")
plt.axis('equal'); plt.grid()

plt.tight_layout()
plt.show()



# 5) EVM (industry standard)

# Error Vector Magnitude: standard communications-industry metric.
# Numerator: RMS error between predicted and true symbols.
# Denominator: RMS magnitude of the true symbols (normalizes for signal scale).
# Lower EVM = better signal quality. Often reported as a percentage.
evm = np.sqrt(np.mean(np.abs(pred_all - true_all)**2) / np.mean(np.abs(true_all)**2))
print("EVM:", evm)



# 6) 16-QAM SLICER + BER

# The 4 valid amplitude levels per axis for 16-QAM
levels = np.array([-3, -1, 1, 3])

def q(v):
    # Snap a real value to its nearest valid 16-QAM level (hard decision)
    return levels[np.argmin(np.abs(levels - v))]

# Apply the slicer to both I and Q of every symbol -> "decided" ideal symbols
pred_dec = np.array([q(r) + 1j*q(i) for r, i in zip(pred_all.real, pred_all.imag)])
true_dec = np.array([q(r) + 1j*q(i) for r, i in zip(true_all.real, true_all.imag)])

# SER: fraction of symbols where predicted decision != true decision
ser = np.mean(pred_dec != true_dec)
print("SER:", ser)

# Gray-coded bit mapping: adjacent levels differ by only 1 bit,
# so single-level slips cause only single-bit errors
mapping = {-3: [0,0], -1: [0,1], 1: [1,1], 3: [1,0]}

def bits(x):
    # Convert an array of decided 16-QAM symbols into their bitstream
    # (2 bits for I + 2 bits for Q per symbol, via Gray coding)
    out = []
    for s in x:
        out += mapping[np.real(s)] + mapping[np.imag(s)]
    return np.array(out)

# BER: fraction of individual bits that differ between predicted and true bitstreams
ber = np.mean(bits(pred_dec) != bits(true_dec))
print("BER:", ber)

In [ ]:
"""
Proper diagnostic — uses the same pipeline as training's `snr` metric.
This tells you exactly what the training measured.

Purpose: independently recompute the SNR metric OUTSIDE of the training loop,
using test-set batches, to sanity-check that the val_snr reported during
training actually reflects real model performance (and wasn't, say, an
artifact of a specific batch or a bug in the Keras metric wiring).
It also reports a "before equalization" baseline (raw channel input) so you
can see how much SNR improvement the neural network is actually providing.
"""
import numpy as np
import tensorflow as tf

N_BATCHES = 20   # Number of test batches to average over, for a stable estimate

mse_before_list, mse_after_list = [], []   # Accumulate per-batch MSE values

for _ in range(N_BATCHES):
    # Pull one batch of (input, target) from the test data generator
    bx, by = next(test_feeder)

    # Run the trained model on the raw input to get its equalized prediction
    bp = model.predict(bx, verbose=0)

    # Convert numpy arrays to TF tensors for use with dsp()/mynorm()/etc.
    bx_tf, by_tf, bp_tf = tf.constant(bx), tf.constant(by), tf.constant(bp)

    # Same transient-skip length used during training (edge samples corrupted
    # by the FIR filter's warm-up region)
    skip = kernel_size // 2

    def process(sig_tf):
        # Reproduce EXACTLY the same preprocessing used inside the `snr` metric
        # during training, so results are directly comparable:
        d = dsp(sig_tf)                              # matched-filter I/Q
        c = tf.complex(d[:, ::2, 0], d[:, ::2, 1])    # downsample to symbol rate, pack complex
        c = c[:, skip:]                               # drop transient edge symbols
        return mynorm(c)                               # normalize to unit peak amplitude

    # Apply the identical pipeline to ground truth, raw input, and model output
    y_true_n   = process(by_tf)   # ground-truth symbols (reference)
    y_before_n = process(bx_tf)   # raw/unequalized input symbols ("before ML")
    y_after_n  = process(bp_tf)   # model-equalized output symbols ("after ML")

    def cpe(sig, ref):
        # Common Phase Error correction: rotate `sig` by the phase that best
        # aligns it with `ref`, so a constant phase offset doesn't inflate error
        tmp = tf.reduce_sum(tf.math.conj(ref) * sig, axis=-1, keepdims=True)
        phi = -tf.atan2(tf.math.imag(tmp), tf.math.real(tmp))
        return sig * tf.exp(tf.complex(0.0, phi))

    # Phase-align both "before" and "after" signals against the true reference
    y_before_cpe = cpe(y_before_n, y_true_n)
    y_after_cpe  = cpe(y_after_n,  y_true_n)

    # Compute mean squared error (at symbol rate) for both cases
    mse_b = tf.reduce_mean(tf.square(tf.abs(y_true_n - y_before_cpe))).numpy()
    mse_a = tf.reduce_mean(tf.square(tf.abs(y_true_n - y_after_cpe))).numpy()

    mse_before_list.append(mse_b)
    mse_after_list.append(mse_a)

# Average MSE across all N_BATCHES test batches, for a more stable/reliable estimate
mse_before = np.mean(mse_before_list)
mse_after  = np.mean(mse_after_list)

# Convert MSE to "SNR" the same way training's `snr` metric does: 1/MSE
# (higher = less error = better)
snr_before = 1.0 / mse_before    # same as training's snr metric
snr_after  = 1.0 / mse_after

# Convert linear SNR to decibels (standard way SNR is reported in comms/DSP)
snr_before_db = 10 * np.log10(snr_before)
snr_after_db  = 10 * np.log10(snr_after)


print("=" * 55)
print("PROPER DIAGNOSTIC (matches training `snr` metric)")
print("=" * 55)
print(f"MSE before equalization : {mse_before:.4f}")
print(f"MSE after  equalization : {mse_after:.4f}")
print(f"SNR before (linear)     : {snr_before:.3f}")
print(f"SNR after  (linear)     : {snr_after:.3f}")
print(f"SNR before (dB)         : {snr_before_db:+.2f} dB")
print(f"SNR after  (dB)         : {snr_after_db:+.2f} dB")
# The dB gain is a clean way to express "how many dB of SNR did the DNN add"
print(f"SNR gain from DNN       : {snr_after_db - snr_before_db:+.2f} dB")
print("=" * 55)
print()
print("→ Compare `SNR after (linear)` with your final val_snr from training")
print("  They should match closely.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

def to_complex(x):
    # Combine last-dim [real, imag] pair into a single complex number
    # (Not actually used below since tf.complex() is used directly instead,
    #  but kept here presumably as a convenience/leftover utility)
    return x[..., 0] + 1j * x[..., 1]

skip = kernel_size // 2   # number of transient/edge symbols to discard after filtering


batch_x, batch_y = next(test_feeder)         # batch_x = channel input, batch_y = ground truth
batch_pred = model.predict(batch_x)           # batch_pred = model's equalized output

# 1) APPLY SAME DSP PIPELINE AS LOSS / METRIC
# Ensures the plotted signals go through IDENTICAL processing to what
# custom_loss/snr used during training, so the plot is a faithful
# visualization of what those metrics actually measured.

#  DSP: matched-filter each signal (I and Q independently)
y_true_dsp   = dsp(batch_y)      # ground truth, matched-filtered
y_pred_dsp   = dsp(batch_pred)   # model output, matched-filtered
y_before_dsp = dsp(batch_x)      # raw channel input, matched-filtered (BEFORE ML baseline)

# Downsample by 2: go from sample-rate (SpS=2) back to symbol-rate
y_true_ds   = y_true_dsp[:, ::2, :]
y_pred_ds   = y_pred_dsp[:, ::2, :]
y_before_ds = y_before_dsp[:, ::2, :]


# Pack the [real, imag] channel pairs into proper complex-valued tensors
y_true_c   = tf.complex(y_true_ds[...,0],   y_true_ds[...,1])
y_pred_c   = tf.complex(y_pred_ds[...,0],   y_pred_ds[...,1])
y_before_c = tf.complex(y_before_ds[...,0], y_before_ds[...,1])


# Drop the first `skip` symbols, which are corrupted by the FIR filter's
# warm-up/edge effect and don't represent steady-state signal quality
y_true_c   = y_true_c[:, skip:]
y_pred_c   = y_pred_c[:, skip:]
y_before_c = y_before_c[:, skip:]

# Scale each signal to unit peak magnitude (via mynorm), so Tx/before/after
# are all on a comparable amplitude scale for plotting/comparison
y_true_n   = mynorm(y_true_c)
y_pred_n   = mynorm(y_pred_c)
y_before_n = mynorm(y_before_c)

# Estimate and remove the constant phase rotation of the model's OUTPUT
# relative to ground truth, so waveform differences plotted are due to
# real distortion/noise, not just an arbitrary phase offset
tmp_after = tf.reduce_sum(tf.math.conj(y_true_n) * y_pred_n, axis=-1, keepdims=True)
phi_after = -tf.atan2(tf.math.imag(tmp_after), tf.math.real(tmp_after))
y_after_cpe = y_pred_n * tf.exp(tf.complex(0.0, phi_after))

# Same phase correction applied to the RAW INPUT relative to ground truth,
# so the "before ML" trace is also fairly aligned for comparison
tmp_before = tf.reduce_sum(tf.math.conj(y_true_n) * y_before_n, axis=-1, keepdims=True)
phi_before = -tf.atan2(tf.math.imag(tmp_before), tf.math.real(tmp_before))
y_before_cpe = y_before_n * tf.exp(tf.complex(0.0, phi_before))


# Select a single example from the batch (index 0) to visualize,
# and pull the underlying numpy arrays out of the TF tensors
idx = 0
tx_seq     = y_true_n[idx].numpy()      # transmitted (ground truth) symbols
rx_before  = y_before_cpe[idx].numpy()  # received symbols BEFORE equalization
rx_after   = y_after_cpe[idx].numpy()   # received symbols AFTER equalization (model output)

# 2) PLOTS – better visual separation
plt.figure(figsize=(14, 5))
n_show = 500  # only plot the first 500 symbols, to keep the plot readable

# Plot the REAL part (I-component) of each signal over symbol index,
# as a time-domain-style waveform comparison (not a constellation scatter)
plt.plot(
    np.real(tx_seq[:n_show]),
    label="Tx ",              # ground truth / transmitted signal
    linestyle="-",
    linewidth=1.5,
)
plt.plot(
    np.real(rx_before[:n_show]),
    label="Before ML ",       # raw channel output, no equalization
    linestyle="--",
    linewidth=1.5,
)
plt.plot(
    np.real(rx_after[:n_show]),
    label="After ML ",        # model-equalized output
    linestyle=":",
    linewidth=2.8,
    marker="o",
    markevery=20,              # only draw a marker every 20 points (avoid clutter)
    markersize=4,
)

plt.title("Tx vs Before ML vs After ML @ 16GBaud")
plt.xlabel("Symbol Index")
plt.ylabel("Amplitude")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

"""
Why this matters beyond SER/BER/EVM: those metrics only tell you about symbol-level accuracy, but they don't reveal how the model is behaving spectrally.
This plot can reveal things like:

Whether the model's output stays within the expected RRC-shaped bandwidth, or leaks power out-of-band (which would matter for adjacent-channel
interference in a real system)
Whether the channel is causing spectral distortion (e.g., bandwidth narrowing from filtering, ripple from chromatic dispersion, etc.) that
"Before ML" shows and "After ML" corrects
Whether the neural network might be doing something spectrally unnatural (e.g., introducing spectral artifacts/noise floor elevation) that wouldn't necessarily
show up clearly in SER or EVM alone
"""

def compute_psd(x, fs, nfft=4096):
    """
    Estimate the Power Spectral Density of a complex baseband signal
    using a single windowed FFT (periodogram method).
    x   : 1D complex np.array
    fs  : sampling frequency (Hz)
    """
    x = x - np.mean(x)  # remove DC offset first, so a nonzero mean doesn't
                         # create a spurious spike at 0 Hz in the spectrum

    # FFT of the signal, zero/truncated to length nfft, then fftshift so
    # frequency axis runs from -fs/2 to +fs/2 (natural baseband layout)
    X = np.fft.fftshift(np.fft.fft(x, nfft))

    # Power spectrum: |FFT|^2, scaled by nfft (periodogram normalization)
    psd = (np.abs(X) ** 2) / nfft

    # Corresponding frequency bins for each FFT sample, also fftshifted
    freqs = np.fft.fftshift(np.fft.fftfreq(nfft, d=1.0/fs))

    # Convert to dB and normalize so the peak of the spectrum sits at 0 dB
    # (makes it easy to visually compare relative shapes across signals
    #  regardless of their absolute power level). 1e-15 avoids log(0).
    psd_db = 10 * np.log10(psd / np.max(psd) + 1e-15)

    return freqs, psd_db


batch_x, batch_y = next(test_feeder)      # batch_x = channel input, batch_y = ground truth
batch_pred = model.predict(batch_x)        # batch_pred = model's equalized output


# Unlike the earlier scripts, this one does NOT downsample to symbol rate
# (no [:, ::2, :]) — it keeps all samples at SpS=2, because you need the
# full sample-rate bandwidth to see the actual occupied spectrum/shape
# (downsampling to symbol rate would alias/hide spectral content beyond
#  the Nyquist bandwidth of one sample per symbol).
y_true_dsp   = dsp(batch_y)     # matched-filtered ground truth,   shape [B, N, 2]
y_before_dsp = dsp(batch_x)     # matched-filtered raw input,      shape [B, N, 2]
y_after_dsp  = dsp(batch_pred)  # matched-filtered model output,   shape [B, N, 2]



idx = 0   # just look at the first example in the batch
tx_seq_2sps = y_true_dsp[idx].numpy()
before_2sps = y_before_dsp[idx].numpy()
after_2sps  = y_after_dsp[idx].numpy()

# Convert each [real, imag] pair into a complex-valued 1D signal at SpS=2
tx_c_2sps     = tx_seq_2sps[:,0] + 1j * tx_seq_2sps[:,1]
before_c_2sps = before_2sps[:,0] + 1j * before_2sps[:,1]
after_c_2sps  = after_2sps[:,0] + 1j * after_2sps[:,1]



# sampfreq = Rs * SpS already defined in your code
# (Rs = symbol/baud rate, SpS = samples per symbol -> fs = actual sample rate)
fs = sampfreq

f_tx,     psd_tx     = compute_psd(tx_c_2sps,     fs)
f_before, psd_before = compute_psd(before_c_2sps, fs)
f_after,  psd_after  = compute_psd(after_c_2sps,  fs)

# Compare spectral shape/occupied bandwidth of: ideal Tx signal,
# raw channel output, and ML-equalized output — all after the same
# RRC matched filter, so differences reflect real spectral distortion
# (e.g. bandwidth narrowing, ripple, out-of-band noise/leakage)
plt.figure(figsize=(10,6))
plt.plot(f_tx/1e9,     psd_tx,     label="Tx (Label)",              linewidth=2)
plt.plot(f_before/1e9, psd_before, label="Before ML (After DSP)",   linestyle="--", linewidth=1.5)
plt.plot(f_after/1e9,  psd_after,  label="After ML (After DSP)",    linestyle=":",  linewidth=1.8)

plt.xlabel("Frequency [GHz]")   # x-axis converted from Hz to GHz for readability
plt.ylabel("Normalized PSD [dB]")
plt.title("PSD Comparison (SpS=2, after RRC DSP)")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

# 16-QAM demapper using Gray 4-PAM per dimension
def qam16_demapper(z):
    """
    z: complex 1D numpy array (symbols)
    returns: 1D numpy array of bits (0/1)
    """
    z = np.asarray(z)

    # For a standard 16-QAM constellation, symbol levels are {±a, ±3a}
    # on each axis, and the *average symbol energy* Es of that
    # constellation is a known constant: Es = 10*a^2 (for equally likely
    # symbols). So instead of assuming a fixed scale, this estimates 'a'
    # directly from the received signal's average power -> makes the
    # demapper self-normalizing / robust to arbitrary input scaling.
    Es = np.mean(np.abs(z)**2)
    a = np.sqrt(Es / 10.0 + 1e-15)   # solve Es = 10*a^2 for a

    re = np.real(z)   # I component of every symbol
    im = np.imag(z)   # Q component of every symbol

    def pam_index(v):
        # Hard-decision slicer for ONE axis (4-PAM), using decision
        # thresholds at -2a, 0, +2a to split the axis into 4 regions,
        # each corresponding to one of the 4 PAM levels: -3a,-a,+a,+3a
        idx = np.zeros_like(v, dtype=int)
        idx[(v >= -2*a) & (v < 0)]  = 1   # region for level -a
        idx[(v >= 0)   & (v < 2*a)] = 2   # region for level +a
        idx[v >= 2*a]              = 3    # region for level +3a
        # (values < -2a fall through to default idx=0, i.e. level -3a)
        return idx

    # Independently decide the PAM level index for I and for Q
    idx_re = pam_index(re)
    idx_im = pam_index(im)

    # Gray-coded bit pairs for the 4 PAM levels, in level order
    # (index 0 -> -3a, 1 -> -a, 2 -> +a, 3 -> +3a), matching the
    # mapping convention used earlier ({-3:[0,0], -1:[0,1], 1:[1,1], 3:[1,0]})
    lut = np.array([[0,0],
                    [0,1],
                    [1,1],
                    [1,0]], dtype=int)

    bits_re = lut[idx_re]   # look up 2 bits for each I decision, shape [N,2]
    bits_im = lut[idx_im]   # look up 2 bits for each Q decision, shape [N,2]

    # Concatenate I-bits and Q-bits per symbol -> 4 bits per 16-QAM symbol
    bits = np.concatenate([bits_re, bits_im], axis=-1)  # [N,4]

    return bits.reshape(-1)  # flatten all symbols' bits into one 1D bitstream

In [ ]:
skip = kernel_size // 2   # number of transient/edge symbols to discard after filtering

def compute_ber_batch(batch_x, batch_y):
    """
    Computes Bit Error Rate for a single batch, both BEFORE equalization
    (raw channel input) and AFTER equalization (model output), using the
    exact same DSP/normalization/CPE pipeline as training's loss & snr metric,
    so the BER numbers are consistent with what was optimized/monitored.
    """

    # Run the trained model on the raw input to get its equalized prediction
    batch_pred = model.predict(batch_x, verbose=0)


    # Matched-filter (RRC) ground truth, raw input, and model output independently
    y_true_dsp   = dsp(batch_y)
    y_before_dsp = dsp(batch_x)
    y_after_dsp  = dsp(batch_pred)

    # Downsample by 2 (same as loss)
    # Go from sample-rate (SpS=2) back down to one sample per symbol
    y_true_ds   = y_true_dsp[:, ::2, :]
    y_before_ds = y_before_dsp[:, ::2, :]
    y_after_ds  = y_after_dsp[:, ::2, :]

    # Complex
    # Pack the [real, imag] channel pairs into complex-valued tensors
    y_true_c   = tf.complex(y_true_ds[...,0],   y_true_ds[...,1])
    y_before_c = tf.complex(y_before_ds[...,0], y_before_ds[...,1])
    y_after_c  = tf.complex(y_after_ds[...,0],  y_after_ds[...,1])

    # Remove transient
    # Drop the first `skip` symbols, corrupted by the FIR filter's warm-up region
    y_true_c   = y_true_c[:, skip:]
    y_before_c = y_before_c[:, skip:]
    y_after_c  = y_after_c[:, skip:]

    # Normalize (same as metric)
    # Scale each signal to unit peak magnitude so all three are on the
    # same amplitude scale before comparison/demapping
    y_true_n   = mynorm(y_true_c)
    y_before_n = mynorm(y_before_c)
    y_after_n  = mynorm(y_after_c)

    # CPE vs true
    # Estimate and remove the constant phase rotation of "before" relative
    # to "true", so BER isn't inflated by an arbitrary phase offset
    tmp_before = tf.reduce_sum(tf.math.conj(y_true_n) * y_before_n, axis=-1, keepdims=True)
    phi_before = -tf.atan2(tf.math.imag(tmp_before), tf.math.real(tmp_before))
    y_before_cpe = y_before_n * tf.exp(tf.complex(0.0, phi_before))

    # Same phase correction applied to "after" (model output) relative to "true"
    tmp_after = tf.reduce_sum(tf.math.conj(y_true_n) * y_after_n, axis=-1, keepdims=True)
    phi_after = -tf.atan2(tf.math.imag(tmp_after), tf.math.real(tmp_after))
    y_after_cpe = y_after_n * tf.exp(tf.complex(0.0, phi_after))

    # Flatten all symbols over batch
    # Collapse (batch, sequence) into one long 1D stream of symbols,
    # since BER is computed over the pooled set of all symbols, not per-sequence
    y_true_flat   = tf.reshape(y_true_n,     [-1]).numpy()
    y_before_flat = tf.reshape(y_before_cpe, [-1]).numpy()
    y_after_flat  = tf.reshape(y_after_cpe,  [-1]).numpy()

    # Demap to bits
    # Convert each complex symbol into its 4 constituent bits via the
    # Gray-coded 16-QAM demapper (hard decision)
    bits_true   = qam16_demapper(y_true_flat)
    bits_before = qam16_demapper(y_before_flat)
    bits_after  = qam16_demapper(y_after_flat)

    # BER
    # Fraction of bits that differ between true and before/after bitstreams
    ber_before = np.mean(bits_true != bits_before)
    ber_after  = np.mean(bits_true != bits_after)

    return ber_before, ber_after

In [ ]:
n_batches = 50   # adjust based on test set size — more batches -> more stable/reliable average

ber_before_list = []   # will collect per-batch BER values (raw, unequalized)
ber_after_list  = []   # will collect per-batch BER values (model-equalized)

# Loop over multiple independent test batches to build up a statistically
# meaningful estimate of BER (a single batch may not contain enough bit
# errors to be a reliable estimate, especially at low BER / high SNR)
for _ in range(n_batches):
    bx, by = next(test_feeder)                    # pull one fresh batch of (input, target)
    ber_b, ber_a = compute_ber_batch(bx, by)       # compute BER before/after for this batch
    ber_before_list.append(ber_b)
    ber_after_list.append(ber_a)

# Average BER across all batches -> single overall performance number
ber_before = np.mean(ber_before_list)
ber_after  = np.mean(ber_after_list)

print(f"Average BER BEFORE ML : {ber_before:.3e}")   # scientific notation, e.g. 1.234e-02
print(f"Average BER AFTER  ML : {ber_after:.3e}")

# Simple plot
# Visualize how BER varies from batch to batch (not just the single
# averaged number), to check consistency/variance across the test set
plt.figure(figsize=(6,4))

# semilogy: log scale on the y-axis, since BER values span many orders
# of magnitude (e.g. 1e-1 down to 1e-6) and would be unreadable on a
# linear scale — this is also the conventional way BER is plotted in comms
plt.semilogy(range(n_batches), ber_before_list, label="Before ML")
plt.semilogy(range(n_batches), ber_after_list,  label="After ML")

plt.xlabel("Batch index")
plt.ylabel("BER (log scale)")
plt.title("BER per batch: Before vs After ML")
plt.grid(True, which="both", linestyle="--", alpha=0.5)  # grid on both major & minor log ticks
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import re
import matplotlib.pyplot as plt

log_file = "/content/drive/MyDrive/MyNewProjectFolder/training_logs.txt"

# Container for ALL training runs found in the log file.
# Each run is a dict of parallel lists: one entry per epoch.
# (Useful if the log file has multiple training sessions appended over time,
#  e.g. from restarting training or re-running the notebook multiple times.)
runs = []  # list of dicts: {"epochs":[], "loss":[], "val_loss":[], "snr":[], "val_snr":[]}

# The run currently being parsed/accumulated
current = {"epochs":[], "loss":[], "val_loss":[], "snr":[], "val_snr":[]}

# Regex to match a Keras-style training log line, e.g.:
# "Epoch 3: loss: 0.0123, snr: 15.2, val_loss: 0.0150, val_snr: 14.8"
# Captures: epoch number, loss, snr, val_loss, val_snr as separate groups
pattern = r"Epoch\s+(\d+):\s+loss:\s+([0-9.eE+-]+),\s+snr:\s+([0-9.eE+-]+),\s+val_loss:\s+([0-9.eE+-]+),\s+val_snr:\s+([0-9.eE+-]+)"

with open(log_file, "r") as f:
    for line in f:
        m = re.search(pattern, line)
        if m:
            # Extract and type-cast each captured value from the regex match
            ep  = int(m.group(1))
            l   = float(m.group(2))
            s   = float(m.group(3))
            vl  = float(m.group(4))
            vs  = float(m.group(5))

            # Detect the start of a NEW training run: if we see "Epoch 1"
            # again after already having collected epochs, that means a
            # previous run just ended and training restarted from scratch.
            # Save the completed run and start a fresh one.
            if ep == 1 and len(current["epochs"]) > 0:
                runs.append(current)
                current = {"epochs":[], "loss":[], "val_loss":[], "snr":[], "val_snr":[]}

            # Append this epoch's values to the current run being built
            current["epochs"].append(ep)
            current["loss"].append(l)
            current["snr"].append(s)
            current["val_loss"].append(vl)
            current["val_snr"].append(vs)

# After the loop ends, the last run in progress hasn't been appended yet
# (since that only happens when a NEW "Epoch 1" is seen) — so append it manually
if len(current["epochs"]) > 0:
    runs.append(current)

# pick the LAST run (or use max length if you prefer)
# Assumes the most recent/relevant training run is the one you want to plot
# (e.g. the final, most up-to-date training session, ignoring earlier aborted ones)
last_run = runs[-1]
epochs   = last_run["epochs"]
loss     = last_run["loss"]
val_loss = last_run["val_loss"]
snr      = last_run["snr"]
val_snr  = last_run["val_snr"]

print(f"Found {len(runs)} runs in log. Plotting last run with {len(epochs)} epochs.")


# Two stacked plots sharing the same x-axis (epoch number):
# top = training/validation loss curves, bottom = training/validation SNR curves
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

# Top: Loss vs Epochs
# Watch for: loss decreasing over time, and whether val_loss tracks
# training loss closely (diverging val_loss = overfitting)
axes[0].plot(epochs, loss,     label="Loss",            color="blue")
axes[0].plot(epochs, val_loss, label="Validation Loss", color="red")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss vs Epochs")
axes[0].grid(True, linestyle="--", alpha=0.5)
axes[0].legend()

# Bottom: SNR vs Epochs
# Since `snr` metric = 1/MSE, this should trend UPWARD as training improves
# (inverse relationship to the loss plot above)
axes[1].plot(epochs, snr,     label="SNR",            color="green")
axes[1].plot(epochs, val_snr, label="Validation SNR", color="orange")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("SNR (metric value)")
axes[1].set_title("SNR vs Epochs")
axes[1].grid(True, linestyle="--", alpha=0.5)
axes[1].legend()

plt.tight_layout()
plt.show()